# 01 — EDA: 30VNFoods

Reads the audit table and the reviewed split manifests produced by `src/data/audit.py`
and `src/data/split.py`. Run the audit and split first; this notebook does not re-split.

Figures are saved to `results/figures/` for the report and website.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.audit import load_csv
from src.data.dataset import get_food_datasets
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD, get_train_transforms, get_val_transforms, get_test_transforms

config = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
RAW_DIR = PROJECT_ROOT / config["data"]["raw_dir"]
MANIFESTS_DIR = PROJECT_ROOT / config["data"]["manifests_dir"]
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

audit = load_csv(MANIFESTS_DIR / "audit.csv")
review = load_csv(MANIFESTS_DIR / "duplicate_review.csv")
splits = {name: load_csv(MANIFESTS_DIR / f"{name}.csv") for name in ("train", "val", "test")}
class_names = sorted({row["class_name"] for row in audit})
print(f"{len(audit):,} image files, {len(class_names)} classes")

## 1. Audit summary: readable images, exclusions and reasons

In [ ]:
readable = sum(row["readable"] == "True" for row in audit)
decisions = Counter(row["decision"] for row in audit)
reasons = Counter(row["reason"].split(" of ")[0] for row in audit if row["decision"] == "exclude")

print(f"Original files : {len(audit):,}")
print(f"Readable       : {readable:,}")
print(f"Decisions      : {dict(decisions)}")
print("Exclusion reasons:")
for reason, count in reasons.most_common():
    print(f"  {count:5d}  {reason}")

print(f"Near-duplicate pairs: {len(review):,}  {dict(Counter(row['decision'] for row in review))}")
groups = Counter(row["group_id"] for row in audit if row["decision"] == "keep")
print(f"Groups with >1 image: {sum(size > 1 for size in groups.values()):,}")

## 2. Class distribution before cleaning

In [ ]:
before = Counter(row["class_name"] for row in audit)
counts = np.array([before[name] for name in class_names])

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(class_names, counts)
ax.set_ylabel("Images")
ax.set_title("Images per class before cleaning")
ax.tick_params(axis="x", rotation=75)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_class_distribution_raw.png", dpi=150)
plt.show()

print(f"min {counts.min()}  max {counts.max()}  mean {counts.mean():.1f}  imbalance ratio (max/min) {counts.max() / counts.min():.2f}")

## 3. Final split sizes and class distribution per split

In [ ]:
total = sum(len(rows) for rows in splits.values())
for name, rows in splits.items():
    print(f"{name:5s}: {len(rows):6,} ({len(rows) / total:.1%})")
print(f"total: {total:6,}  (original {len(audit):,}, removed {len(audit) - total:,})")
assert len(splits["train"]) >= 5000, "Handbook 19.1 requires at least 5,000 training samples"

per_split = {name: Counter(row["class_name"] for row in rows) for name, rows in splits.items()}
x = np.arange(len(class_names))
fig, ax = plt.subplots(figsize=(14, 5))
for offset, name in zip((-0.27, 0, 0.27), splits):
    ax.bar(x + offset, [per_split[name][c] for c in class_names], width=0.27, label=name)
ax.set_xticks(x, class_names, rotation=75)
ax.set_ylabel("Images")
ax.set_title("Images per class in each split")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_class_distribution_splits.png", dpi=150)
plt.show()

for name in ("val", "test"):
    smallest = min(per_split[name][c] for c in class_names)
    print(f"smallest class in {name}: {smallest}")

## 4. Image size and aspect ratio

In [ ]:
kept = [row for row in audit if row["decision"] == "keep"]
widths = np.array([int(row["width"]) for row in kept])
heights = np.array([int(row["height"]) for row in kept])
aspect = widths / heights

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(widths, bins=50); axes[0].set_title("Width (px)")
axes[1].hist(heights, bins=50); axes[1].set_title("Height (px)")
axes[2].hist(aspect, bins=50); axes[2].set_title("Aspect ratio (w/h)")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_image_sizes.png", dpi=150)
plt.show()

print(f"width  min {widths.min()}  median {np.median(widths):.0f}  max {widths.max()}")
print(f"height min {heights.min()}  median {np.median(heights):.0f}  max {heights.max()}")
print(f"smaller than 224 on the short side: {np.sum(np.minimum(widths, heights) < 224):,}")
print("formats:", dict(Counter(row["format"] for row in kept)))
print("modes  :", dict(Counter(row["mode"] for row in kept)))

## 5. One example per class

In [ ]:
first_by_class = {}
for row in splits["train"]:
    first_by_class.setdefault(row["class_name"], row)

fig, axes = plt.subplots(5, 6, figsize=(15, 13))
for ax, name in zip(axes.flat, class_names):
    with Image.open(RAW_DIR / first_by_class[name]["path"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(name, fontsize=9)
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_class_examples.png", dpi=120)
plt.show()

## 6. Problematic images: duplicates and reviewed label conflicts

In [ ]:
confirmed = [row for row in review if row["decision"] != "different"]
pairs = [row for row in confirmed if row["class_a"] != row["class_b"]][:4] or confirmed[:4]

if pairs:
    fig, axes = plt.subplots(len(pairs), 2, figsize=(6, 3 * len(pairs)), squeeze=False)
    for (ax_a, ax_b), row in zip(axes, pairs):
        for ax, key, label in ((ax_a, "image_a", "class_a"), (ax_b, "image_b", "class_b")):
            with Image.open(RAW_DIR / row[key]) as image:
                ax.imshow(image.convert("RGB"))
            ax.set_title(f"{row[label]}\nd={row['distance']} mae={row['mae']} {row['decision']}", fontsize=8)
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "eda_duplicate_examples.png", dpi=120)
    plt.show()
else:
    print("No near-duplicate pairs found.")

## 7. A preprocessed training batch

In [ ]:
train_dataset, _, _ = get_food_datasets(
    MANIFESTS_DIR, RAW_DIR, get_train_transforms(), get_val_transforms(), get_test_transforms(),
)
rng = np.random.default_rng(42)
indices = rng.choice(len(train_dataset), size=16, replace=False)
mean, std = np.array(IMAGENET_MEAN), np.array(IMAGENET_STD)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, index in zip(axes.flat, indices):
    image, label = train_dataset[int(index)]
    print_shape = image.shape
    picture = np.clip(image.permute(1, 2, 0).numpy() * std + mean, 0, 1)
    ax.imshow(picture)
    ax.set_title(train_dataset.classes[label], fontsize=8)
    ax.axis("off")
fig.suptitle(f"Training batch after augmentation, tensor shape {tuple(print_shape)}")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_preprocessed_batch.png", dpi=120)
plt.show()

## 8. Leakage checks across splits

In [ ]:
by_id = {row["image_id"]: row for row in audit}

for key in ("image_id", "group_id"):
    sets = {name: {row[key] for row in rows} for name, rows in splits.items()}
    overlap = (sets["train"] & sets["val"]) | (sets["train"] & sets["test"]) | (sets["val"] & sets["test"])
    print(f"{key:8s} shared between splits: {len(overlap)}")

sha_sets = {name: {by_id[row["image_id"]]["sha256"] for row in rows} for name, rows in splits.items()}
overlap = (sha_sets["train"] & sha_sets["val"]) | (sha_sets["train"] & sha_sets["test"]) | (sha_sets["val"] & sha_sets["test"])
print(f"sha256   shared between splits: {len(overlap)}")

## 9. Notes (fill in after running)

- Bias: e.g. photo sources, backgrounds, plating style, lighting.
- Limits: near-duplicate detection (dHash, distance <= 8) misses heavy crops/rotations.
- Remaining leakage risk: the same dish photographed from different angles in one restaurant.